# Content
- [Bayesian Knowledge Tracing](#Bayesian-Knowledge-Tracing)
- [A simple BKT Implementation](#A-simple-BKT-Implementation)
- [Optional: try out pyBKT!](#Optional:-try-out-pyBKT!)

# Bayesian Knowledge Tracing

Bayesian Knowledge Tracing (BKT) is used to model student mastery of skills.  
Expectation-Maximization (EM) algorithm is used to estimate the parameters of the BKT model for a given skill.

### Model Parameters:

- $P(L_0)$: – the probability of the student knowing the skill beforehand.
- $P(T)$: – is used to compute the probability a student will master a skill at each answer opportunity.
- $P(S)$: – represents the probability of the student solving a problem without having mastered the skill.
- $P(G)$: – represents the probability of the student making a mistake applying a skill they have mastered.

### Bayesian Update:

Given a student's current knowledge $P(L)$ and whether they answered correctly ($correct = 1$) or not ($correct = 0$):

- Predict probability of a correct response:

  $$
  P(\text{Correct}) = P(L) \cdot (1 - P(S)) + (1 - P(L)) \cdot P(G)
  $$

- After observing the student's response, update knowledge estimate:

  - If correct:
  
    $$
    P(L|Correct) = \frac{P(L) \cdot (1 - P(S))}{P(\text{Correct})}
    $$

  - If incorrect:
  
    $$
    P(L|Incorrect) = \frac{P(L) \cdot P(S)}{P(L) \cdot P(S) + (1 - P(L)) \cdot (1 - P(G))}
    $$

- Apply learning (transition) step:

  $$
  P(L) \leftarrow P(L|obs) + (1 - P(L|obs)) \cdot P(T)
  $$

### EM Procedure:

1. **E-step**: For each student and observation, compute the posterior knowledge state $P(L∣obs)$ using current parameters.
2. **M-step**: Update parameters based on observed transitions and outcomes.
3. Repeat until parameters converge (changes are below a small tolerance).


# A simple BKT Implementation

### (1) Imports and Load Data

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, f1_score, precision_score, recall_score

data_1 = "https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/as.csv"
data_2 = "https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/ct.csv"

df_1 = pd.read_csv(data_1, encoding='latin', low_memory=False)
df_2 = pd.read_csv(data_2, encoding='latin')

df_2.head(5)

,Unnamed: 0,Row,Anon Student Id,Problem Hierarchy,Problem Name,Problem View,Step Name,Step Start Time,First Transaction Time,Correct Transaction Time,Step End Time,Step Duration (sec),Correct Step Duration (sec),Error Step Duration (sec),Correct First Attempt,Incorrects,Hints,Corrects,KC(Default),Opportunity(Default)
0,1576,1927,745Yh,"Unit RATIO-PROPORTION, Section RATIO-PROPORTION-2",RATIO2-001,1,SimplifiedNumeratorQuantity1,2006-11-14 10:18:00.0,2006-11-14 10:18:05.0,2006-11-14 10:18:05.0,2006-11-14 10:18:05.0,5.0,5.0,NaN,1,0,0,1,Calculate unit rate,1
1,1580,1931,745Yh,"Unit RATIO-PROPORTION, Section RATIO-PROPORTION-2",RATIO2-001,1,SimplifiedNumeratorQuantity2,2006-11-14 10:18:11.0,2006-11-14 10:18:17.0,2006-11-14 10:18:34.0,2006-11-14 10:18:34.0,23.0,NaN,23.0,0,1,0,1,Calculate unit rate,2
2,1596,1947,745Yh,"Unit RATIO-PROPORTION, Section RATIO-PROPORTION-2",RATIO2-012,1,SimplifiedNumeratorQuantity1,2006-11-14 10:50:52.0,2006-11-14 10:50:57.0,2006-11-14 10:51:11.0,2006-11-14 10:51:11.0,18.0,NaN,18.0,0,1,0,1,Calculate unit rate,3
3,1597,1948,745Yh,"Unit RATIO-PROPORTION, Section RATIO-PROPORTION-2",RATIO2-012,1,SimplifiedNumeratorQuantity2,2006-11-14 10:51:11.0,2006-11-14 10:51:14.0,2006-11-14 10:51:14.0,2006-11-14 10:51:14.0,3.0,3.0,NaN,1,0,0,1,Calculate unit rate,4
4,1612,1963,745Yh,"Unit RATIO-PROPORTION, Section RATIO-PROPORTION-2",RATIO2-054,1,SimplifiedNumeratorQuantity2,2006-11-28 09:53:43.0,2006-11-28 09:53:47.0,2006-11-28 09:53:56.0,2006-11-28 09:53:56.0,13.0,NaN,13.0,0,1,0,1,Calculate unit rate,5


avaiable dataset: 

In [7]:
print(df_2.columns)

Index(['Unnamed: 0', 'Row', 'Anon Student Id', 'Problem Hierarchy',
       'Problem Name', 'Problem View', 'Step Name', 'Step Start Time',
       'First Transaction Time', 'Correct Transaction Time', 'Step End Time',
       'Step Duration (sec)', 'Correct Step Duration (sec)',
       'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects',
       'Hints', 'Corrects', 'KC(Default)', 'Opportunity(Default)'],
      dtype='object')


### (2) Bayesian update student knowledge 

In [13]:
def bayesian_update(P_L, correct, P_S, P_G, P_T):
    p_correct = P_L * (1 - P_S) + (1 - P_L) * P_G
    if correct == 1:
        numer = P_L * (1 - P_S)
        denom = p_correct
    else:
        numer = P_L * P_S
        denom = P_L * P_S + (1 - P_L) * (1 - P_G)

    p_L_given_obs = numer / denom if denom > 0 else P_L
    
    '''
    TODO: 
            1. Calculate p_L_updated
            2. return values of p_L_updated and p_correct
    '''
    raise NotImplementedError

### (3) Update parameters

In [14]:
def update_parameters(df_skill, p_L_values, learning_rate=0.5):
    P_L0 = np.mean([p[0] for p in p_L_values]) 
    P_T = np.mean([p[1] for p in p_L_values])
    P_S = np.mean([p[2] for p in p_L_values]) 
    P_G = np.mean([p[3] for p in p_L_values])   
    #print(f"Parameter updates - P_T: {P_T}, P_S: {P_S}, P_G: {P_G}")

    P_T = P_T + learning_rate * (np.random.uniform(0.15, 0.25) - P_T)
    P_S = P_S + learning_rate * (np.random.uniform(0.15, 0.25) - P_S)
    P_G = P_G + learning_rate * (np.random.uniform(0.15, 0.25) - P_G)

    return P_L0, P_T, P_S, P_G

### (4) Expectation-Maximization process

In [18]:
def bkt_em(df, max_iter=15, tolerance=1e-4):
    skills = df['KC(Default)'].unique()
    all_params = {}

    for skill in skills:
        print(f"Training for skill: {skill}")
        df_skill = df[df['KC(Default)'] == skill].copy()
        df_skill.sort_values(by=['Anon Student Id', 'Step Start Time'], inplace=True)

        P_L0 = 0.2  # Initial guess for knowledge
        P_T = 0.2   # Initial guess for learning rate
        P_S = 0.2   # Initial guess for slip
        P_G = 0.2   # Initial guess for guess rate

        p_L_values = []
        prev_params = [P_L0, P_T, P_S, P_G]

        for iteration in range(max_iter):
            # E-step: Estimate student knowledge (P(L)) and predict correctness
            predictions = []
            ground_truths = []
            for student, group in df_skill.groupby('Anon Student Id'):
                P_L = P_L0
                for _, row in group.iterrows():
                    correct = row['Correct First Attempt']
                    P_L, p_correct = bayesian_update(P_L, correct, P_S, P_G, P_T)
                    predictions.append(p_correct)
                    ground_truths.append(correct)  
                p_L_values.append([P_L, P_T, P_S, P_G])

            # M-step: Update parameters using the estimated knowledge (P(L))
            P_L0, P_T, P_S, P_G = update_parameters(df_skill, p_L_values)
            #print(f"Iteration {iteration + 1}: P(L0): {P_L0:.3f}, P(T): {P_T:.3f}, P(S): {P_S:.3f}, P(G): {P_G:.3f}")
            param_diff = np.abs(np.array([P_L0, P_T, P_S, P_G]) - np.array(prev_params))
            if np.all(param_diff < tolerance):
                print(f"Converged at iteration {iteration + 1}. Parameters changed by less than {tolerance}.")
                break
            prev_params = [P_L0, P_T, P_S, P_G]
        all_params[skill] = {'P_L0': P_L0, 'P_T': P_T, 'P_S': P_S, 'P_G': P_G}

    return all_params

### (5) Run BKT on skills

In [19]:
all_skill_params = bkt_em(df_2)
for skill, params in all_skill_params.items():
    print(f"Final learned parameters for {skill}:")
    print(f"P(L0) = {params['P_L0']:.3f}, P(T) = {params['P_T']:.3f}, P(S) = {params['P_S']:.3f}, P(G) = {params['P_G']:.3f}")

Training for skill: Calculate unit rate
Training for skill: Calculate part in proportion with fractions
Training for skill: Calculate total in proportion with fractions
Training for skill: Plot whole number
Training for skill: Plot terminating proper fraction
Training for skill: Plot imperfect radical
Training for skill: Plot non-terminating improper fraction
Training for skill: Plot pi
Training for skill: Plot decimal - thousandths
Training for skill: Finding the intersection, SIF
Training for skill: Finding the intersection, Mixed
Training for skill: Finding the intersection, GLF
Final learned parameters for Calculate unit rate:
P(L0) = 0.985, P(T) = 0.207, P(S) = 0.190, P(G) = 0.173
Final learned parameters for Calculate part in proportion with fractions:
P(L0) = 0.956, P(T) = 0.199, P(S) = 0.207, P(G) = 0.207
Final learned parameters for Calculate total in proportion with fractions:
P(L0) = 0.946, P(T) = 0.194, P(S) = 0.218, P(G) = 0.179
Final learned parameters for Plot whole numb

### (6) BKT Prediction

In [17]:
train_df, test_df = train_test_split(df_2, test_size=0.2, random_state=42)

def BKT_predict(test_df, all_params):
    """
    Predicts correctness for the test data using the learned parameters (P(L0), P(T), P(S), P(G))
    and computes evaluation metrics (RMSE, F1 Score, Precision, Recall).
    """
    predictions = []
    ground_truths = []
    
    for student, group in test_df.groupby('Anon Student Id'):
        P_L = {} 
        for _, row in group.iterrows():
            skill = row['KC(Default)']
            correct = row['Correct First Attempt']
            if skill in all_params:
                P_L0, P_T, P_S, P_G = all_params[skill]['P_L0'], all_params[skill]['P_T'], all_params[skill]['P_S'], all_params[skill]['P_G']
            else:
                P_L0, P_T, P_S, P_G = 0.2, 0.2, 0.2, 0.2
            
            if skill not in P_L:
                P_L[skill] = P_L0

            p_correct = P_L[skill] * (1 - P_S) + (1 - P_L[skill]) * P_G
            predictions.append(p_correct)
            ground_truths.append(correct)
            P_L[skill], _ = bayesian_update(P_L[skill], correct, P_S, P_G, P_T)
            

    rmse = np.sqrt(mean_squared_error(ground_truths, predictions))
    f1 = f1_score(ground_truths, np.round(predictions))
    precision = precision_score(ground_truths, np.round(predictions))
    recall = recall_score(ground_truths, np.round(predictions))
    
    return rmse, f1, precision, recall


rmse, f1, precision, recall = BKT_predict(test_df, all_skill_params)


print("BKT Results:")
print("RMSE:", rmse)
print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)

BKT Results:
RMSE: 0.491699052743496
F1 Score: 0.7660601559598961
Precision: 0.6228864734299517
Recall: 0.9946962391513983


# Optional: try out pyBKT!

pyBKT is a python library of the Bayesian Knowledge Tracing algorithm and variants, estimating student cognitive mastery from problem solving sequences.

Install pyBKT and import their model

In [ ]:
# Install pyBKT from pip!
!pip install pyBKT
from pyBKT.models import Model

The following is an example for usage 

In [ ]:
# this cell is a sumary of key features from pyBKT, just for reference (you will not be able to modify this cell)
# import necessary libraries (!pip install package-name)
# choose and explore the dataset either data_1 or data_2

def mae(true_vals, pred_vals):
  """ Calculates the mean absolute error. """
  return np.mean(np.abs(true_vals - pred_vals))

# This code fetches data, fits, predicts, evaluates and crossvalidates
# a BKT model on all skills in Cognitive Tutor. It uses the mean absolute
# error as the desired error metric.

model = Model(seed = 42, num_fits = 5)
model.fit(data_path = '') 
preds_df = model.predict(data_path = '')
mae_error = model.evaluate(data_path = '', metric = mae)
cv_errors = model.crossvalidate(data_path = '', metric = mae)
model.fit(data_path = '')
params_df = model.params()
print("Training MAE: %f" % mae_error)
cv_errors

Try it yourself using the available data. If you need to import external libraries, you can install them using:  
`!pip install package-name`

### You will not be able to delete or add any new cells. Please only use the following three empty cells for implementation.

In [ ]:
## Your implementation

In [ ]:
## Your implementation

In [ ]:
## Your implementation